# La Física del Disco y la Snowline Dinámica
En este notebook profundizaremos en cómo PA3Py modela la migración térmica de la Snowline utilizando los resultados de transferencia radiativa de **Oka et al. (2011)** combinados con la evolución de la tasa de acreción de gas de **Hartmann et al. (1998)**.


In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.interpolate import interp1d

# Añadir PA3Py al path local
sys.path.insert(0, os.path.abspath('../../src'))
from pa3py import PA3Py
from pa3py.snowline import generate_rsnow_array, mdot_time, _csv_path
from pa3py import constants as c

# Estilo de Plotting (Paper)
plt.rcParams.update({
    'text.usetex': False,
    'font.family': 'serif',
    'font.size': 14,
    'axes.labelsize': 16,
    'lines.linewidth': 2.0,
    'figure.dpi': 150,
})


## 1. Evolución de la Tasa de Acreción (Hartmann et al.)
Asumimos que la tasa de acreción del gas decae en el tiempo como $\dot{M}(t) = \dot{M}_0 (t / 1 \text{Myr})^{-\eta}$.


In [ ]:
t_eval = np.logspace(-1, 1, 500) # 0.1 a 10 Myr
eta_plot = 1.5
m_t = mdot_time(t_eval, eta=eta_plot)
log_t_eval = np.log10(t_eval)
log_m_t = np.log10(m_t)


## 2. Interpolación de la Snowline (Oka et al. 2011)
Cargamos los datos corregidos de Oka et al. 2011 y generamos una interpolación log-log para mapear cualquier $\dot{M}$ a su respectiva $r_{snow}$.


In [ ]:
mdot_list, rsnow_list = [], []
with open(_csv_path, 'r') as f:
    lines = f.readlines()
    for line in lines[1:]:
        line = line.strip()
        if not line: continue
        parts = line.split(';')
        if len(parts) == 2:
            mdot_list.append(float(parts[0]))
            rsnow_list.append(float(parts[1]))

mdot_raw, rsnow_raw = np.array(mdot_list), np.array(rsnow_list)
sort_idx = np.argsort(mdot_raw)
mdot_unique, unique_indices = np.unique(mdot_raw[sort_idx], return_index=True)
rsnow_unique = rsnow_raw[sort_idx][unique_indices]

interp_log_rsnow = interp1d(np.log10(mdot_unique), np.log10(rsnow_unique), kind='linear', fill_value='extrapolate')

def r_snow_from_mdot(mdot_val):
    return 10**interp_log_rsnow(np.log10(mdot_val))

def r_snow_time(t_myr, eta=1.5):
    return r_snow_from_mdot(mdot_time(t_myr, eta))


## 3. Diagnóstico Completo (El Gráfico Oficial)
Construimos un panel triple mostrando toda la física de la snowline.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
ax1, ax2, ax3 = axes
color_main, color_raw = '#d95f02', '#1f78b4'

# --- PANEL 1: Evolución de Mdot(t) ---
ax1.plot(log_t_eval, log_m_t, color=color_main, linewidth=2.5, label=fr'$\eta = {eta_plot}$')
ax1.set_xlim(-1, 1)
ax1.set_xticks([-1, -0.5, 0, 0.5, 1])
ax1.set_xlabel(r'$\log_{10}(t\ [\mathrm{Myr}])$')
ax1.set_ylabel(r'$\log_{10}(\dot{M}\ [M_\odot\ \mathrm{yr}^{-1}])$')
ax1.legend(loc='upper right', framealpha=0.85)

# --- PANEL 2: Interpolación Oka et al ---
m_test = np.logspace(-12, -7, 500)
r_test = r_snow_from_mdot(m_test)
ax2.plot(np.log10(mdot_raw), rsnow_raw, color=color_raw, linestyle='-', linewidth=3, alpha=0.5, label='Oka et al. 2011')
ax2.plot(np.log10(m_test), r_test, color=color_main, linestyle='--', linewidth=2.5, label='Interpolación')
ax2.set_xlim(-7, -12)
ax2.set_yscale('log')
ax2.set_ylim(0.1, 10)
ax2.set_yticks([0.1, 1, 10])
ax2.get_yaxis().set_major_formatter(ticker.ScalarFormatter())
ax2.set_xlabel(r'$\log_{10}(\dot{M}\ [M_\odot\ \mathrm{yr}^{-1}])$')
ax2.set_ylabel(r'$r_{\mathrm{snow}}\ [\mathrm{AU}]$')

# Linea en 10 Myr
log_m_10myr = np.log10(mdot_time(10, eta=eta_plot))
ax2.axvline(log_m_10myr, color='k', linestyle=':', linewidth=1.5, alpha=0.6, label=r'$\dot{M}(10\,\mathrm{Myr})$')
ax2.legend(loc='lower right', framealpha=0.85)

# --- PANEL 3: Evolución de r_snow(t) ---
ax3.plot(log_t_eval, r_snow_time(t_eval, eta=eta_plot), color=color_main, linewidth=2.5, label=fr'$\eta = {eta_plot}$')
ax3.set_xlim(-1, 1)
ax3.set_xticks([-1, -0.5, 0, 0.5, 1])
ax3.set_yscale('log')
ax3.set_ylim(0.1, 10)
ax3.set_yticks([0.1, 1, 10])
ax3.get_yaxis().set_major_formatter(ticker.ScalarFormatter())
ax3.set_xlabel(r'$\log_{10}(t\ [\mathrm{Myr}])$')
ax3.set_ylabel(r'$r_{\mathrm{snow}}\ [\mathrm{AU}]$')

from scipy.optimize import brentq
try:
    t_1au = brentq(lambda t: r_snow_time(t, eta=eta_plot) - 1.0, 0.1, 10.0)
    ax3.plot(np.log10(t_1au), 1.0, 'ko', markersize=6, zorder=5, label=f'{t_1au:.2f} Myr')
    ax3.axvline(np.log10(t_1au), color='k', linestyle=':', alpha=0.6)
    ax3.axhline(1.0, color='k', linestyle=':', alpha=0.6)
except ValueError:
    pass
ax3.legend(loc='lower left', framealpha=0.85)

for ax in [ax1, ax2, ax3]:
    ax.grid(True, which='both', linestyle=':', alpha=0.3, color='gray')
plt.show()
